In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", None)

In [ ]:
df = pd.read_csv("../data/raw/telco.csv")
df.head()

In [ ]:
df.shape
df.info()
df.describe()

Les statistiques descriptives révèlent plusieurs éléments clés :
- `tenure` : moyenne de 32 mois (std=24), indiquant une base client hétérogène
  avec des profils très récents et très anciens
- `MonthlyCharges` : moyenne de 64€ (std=30), avec une forte dispersion
  suggérant des offres très variées
- `TotalCharges` : très corrélé à tenure × MonthlyCharges, à surveiller
  pour éviter la multicolinéarité

In [ ]:
df["Churn"].value_counts()
df["Churn"].value_counts(normalize=True)

In [ ]:
df.isnull().sum()

In [ ]:
df["TotalCharges"].dtype

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

In [ ]:
df["TotalCharges"].dtype

In [ ]:
df.isnull().sum()

In [ ]:
df["TotalCharges"].unique()[:20]

In [ ]:
df["TotalCharges"].isnull().sum()

In [ ]:
df[df["TotalCharges"].isnull()]["tenure"].value_counts()

Les 11 valeurs manquantes dans `TotalCharges` correspondent toutes à des clients
avec un tenure de 0 mois, c'est-à-dire des clients n'ayant généré aucune facturation.
Ces lignes sont supprimées car elles n'apportent pas d'information exploitable pour
la modélisation du churn.

In [ ]:
df["Churn"].value_counts()
df["Churn"].value_counts(normalize=True)

Environ 26% des clients quittent le service.

Ce taux de churn est significatif et justifie la mise en place d’un modèle prédictif afin d’anticiper les départs.

In [ ]:
df = df.dropna()

In [ ]:
sns.countplot(x="Contract", hue="Churn", data=df)
plt.show()

Les clients avec un contrat mensuel présentent un taux de churn significativement plus élevé que ceux engagés sur un ou deux ans.

Cela suggère que l’engagement contractuel joue un rôle clé dans la rétention client.

In [ ]:
sns.boxplot(x="Churn", y="tenure", data=df)
plt.show()

Les clients ayant une ancienneté faible (tenure bas) sont plus susceptibles de churner.

Cela indique que les premières phases du cycle de vie client sont critiques pour la rétention.

In [ ]:
sns.boxplot(x="Churn", y="MonthlyCharges", data=df)
plt.show()

Les clients ayant des charges mensuelles élevées présentent un taux de churn plus important.

Cela suggère une sensibilité au prix ou une inadéquation entre le coût et la valeur perçue.

In [ ]:
num_cols = df.select_dtypes(include="number").columns
plt.figure(figsize=(8, 5))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Matrice de corrélation — variables numériques")
plt.show()

In [ ]:
cat_cols = ["InternetService", "PaymentMethod", "Contract", "TechSupport"]
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, col in zip(axes.flatten(), cat_cols, strict=False):
    churn_rate = df.groupby(col)["Churn"].value_counts(normalize=True).unstack()
    churn_rate["Yes"].sort_values().plot(kind="bar", ax=ax, color="salmon")
    ax.set_title(f"Taux de churn par {col}")
    ax.set_ylabel("Taux de churn")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

L'analyse des variables catégorielles confirme plusieurs facteurs de churn :
- **InternetService** : les clients avec fibre optique churne davantage
  que ceux avec DSL ou sans internet
- **PaymentMethod** : le paiement par chèque électronique est associé
  à un taux de churn plus élevé
- **Contract** : confirme l'analyse précédente, le contrat mensuel
  est le facteur le plus discriminant
- **TechSupport** : les clients sans support technique churne plus,
  suggérant un lien entre satisfaction et rétention

## Déséquilibre de classes

Le dataset présente un déséquilibre modéré : 73% non-churn vs 27% churn.
Ce déséquilibre peut biaiser le modèle vers la classe majoritaire.

Trois stratégies seront évaluées lors de la phase de modélisation :
- **class_weight='balanced'** : ajuste automatiquement les poids dans
  Scikit-learn, sans modifier les données
- **SMOTE** : génère des exemples synthétiques de la classe minoritaire
  pour rééquilibrer le dataset d'entraînement
- **Ajustement du seuil de décision** : abaisser le seuil de 0.5
  pour augmenter le recall sur la classe churn

Le choix final sera guidé par la métrique **F1-score** plutôt que
l'accuracy, plus adaptée aux datasets déséquilibrés.

## Synthèse des insights

L’analyse met en évidence plusieurs facteurs clés influençant le churn :

- Les contrats mensuels sont fortement associés à un taux de churn élevé
- Les nouveaux clients sont plus susceptibles de quitter le service
- Des charges mensuelles élevées augmentent le risque de churn

Ces éléments permettent d’orienter la construction du modèle prédictif et les stratégies de rétention client.